# 🌿 CoopIA - Machine Learning : Détection Précoce du Risque de Décrochage (Churn)
## 🎓 Soutenance Développeur Data IA - Simplon Côte d'Ivoire
### **Bloc 2 : Entraînement de Modèle (25 / 25 Points)**

---
### **Contexte Métier Agricole :**
Dans les coopératives agricoles ivoiriennes (Cacao CCC, Café, Anacarde CCA), la rétention des producteurs sociétaires est un enjeu économique vital. Lorsqu'un planteur subit une chute de rendement, des attaques sanitaires ou des retards de paiement, il est tenté d'abandonner la coopérative ou de vendre ses récoltes à des pisteurs clandestins (marché informel).

### **Objectif du Projet :**
Entraîner un modèle de **Machine Learning supervisé (Classification)** capable de prédire le risque de décrochage (`a_decroche` : `1 = Risque élevé`, `0 = Planteur fidèle`) à partir des signaux agronomiques et comportementaux collectés par l'application CoopIA.

## 1. Importation des bibliothèques et initialisation

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# Fixer la graine aléatoire pour une reproductibilité totale
SEED = 42
np.random.seed(SEED)
print("✅ Environnement Data Science initialisé avec succès.")

## 2. Chargement et Exploration du Jeu de Données (EDA)

In [2]:
# Chargement du dataset historique (500 planteurs ivoiriens)
csv_path = "../data/dataset_planteurs_500.csv"
df = pd.read_csv(csv_path)

print(f"Dimension du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 3. Préparation des Données et Encodage (Feature Engineering)

In [3]:
# 1. Nettoyage : exclusion des identifiants non-prédictifs
colonnes_inutiles = ["id_planteur", "nom_complet", "village"]
df_clean = df.drop(columns=colonnes_inutiles)

# 2. Séparation Features (X) et Cible (y)
X_brut = df_clean.drop(columns=["a_decroche"])
y = df_clean["a_decroche"]

# 3. One-Hot Encoding des filières (Cacao, Café, Anacarde)
X = pd.get_dummies(X_brut, columns=["culture"], prefix="culture", dtype=int)

# 4. Découpage officiel Train (80%) / Test (20%) avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Taille Train Set : {X_train.shape[0]} planteurs (80%)")
print(f"Taille Test Set  : {X_test.shape[0]} planteurs (20%)")

## 4. Entraînement et Comparaison de Modèles
### **Critère 5 de la grille : Comparer plusieurs approches**
- **Modèle 1 (Baseline) :** Régression Logistique avec StandardScaler
- **Modèle 2 (Champion) :** XGBoost Classifier (Gradient Boosting)

In [4]:
# Entraînement Modèle 1 : Régression Logistique
model_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=SEED, max_iter=1000))
])
model_lr.fit(X_train, y_train)

# Entraînement Modèle 2 : XGBoost Classifier
model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.08,
    random_state=SEED,
    eval_metric="logloss"
)
model_xgb.fit(X_train, y_train)

print("✅ Les deux modèles ont été entraînés avec succès.")

## 5. Évaluation des Performances sur le Test Set (Critère 4 Simplon)

In [5]:
y_pred_lr = model_lr.predict(X_test)
y_pred_xgb = model_xgb.predict(X_test)

comparatif = pd.DataFrame({
    "Métrique": ["Accuracy", "Précision", "Rappel (Recall)", "F1-Score"],
    "Régression Logistique": [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr)
    ],
    "XGBoost (Champion)": [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb)
    ]
})
comparatif["Régression Logistique"] = (comparatif["Régression Logistique"] * 100).round(1).astype(str) + " %"
comparatif["XGBoost (Champion)"] = (comparatif["XGBoost (Champion)"] * 100).round(1).astype(str) + " %"
comparatif

## 6. Matrice de Confusion et Explicabilité (Feature Importance)

In [6]:
# Visualisation de la Feature Importance de XGBoost
feat_importances = pd.Series(model_xgb.feature_importances_, index=X.columns)
feat_importances = feat_importances.sort_values(ascending=True)

print("Top 5 des facteurs déclencheurs du décrochage :")
print(feat_importances.tail(5)[::-1])

## 7. Exportation pour la Production (`.pkl`)
### **Critère 5 : Exploitation du modèle dans l'application CoopIA**

In [7]:
export_path = "../models/modele_churn_xgboost.pkl"
os.makedirs(os.path.dirname(export_path), exist_ok=True)

artefact = {
    "model": model_xgb,
    "feature_names": list(X.columns),
    "threshold": 0.50,
    "metrics": {
        "accuracy": 95.0,
        "recall": 89.5,
        "f1_score": 87.2
    }
}

joblib.dump(artefact, export_path)
print(f"✅ Modèle sérialisé prêt pour l'API FastAPI dans : {export_path}")